# NITRATES non-imaging quickstart (local setup)  
This short notebook sets up a small, laptop-friendly response/data bundle that ships with the repository (`tests/nitrates_resp_dir`) and previews the trigger metadata we will analyze in the follow-up notebook.  

**Why this data set?** The directory already contains trimmed responses, background estimates, and event files, so you can run the full non-imaging likelihood example without downloading the multi-GB response bundles used on the cluster.

## 1) Point to the bundled responses
Set `NITRATES_RESP_DIR` to the lightweight test bundle. If you have already exported the variable, the existing value is used instead. Run this cell from the repository root or the `notebooks/` directory.

In [ ]:
import os
from pathlib import Path

repo_root = Path.cwd()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent
resp_dir = Path(os.environ.get('NITRATES_RESP_DIR', repo_root/'tests'/'nitrates_resp_dir')).resolve()
os.environ['NITRATES_RESP_DIR'] = str(resp_dir)

print(f"Repository root: {repo_root}")
print(f"Using NITRATES_RESP_DIR: {resp_dir}")
print()
print('Contents:')
for p in sorted(resp_dir.iterdir()):
    print(' -', p.name)


## 2) Inspect the trigger metadata
The bundled `results.db` already contains the trigger time and analysis bookkeeping that we will reuse. The helper functions in `nitrates.lib` provide a thin wrapper around the SQLite file.

In [ ]:
import numpy as np
from nitrates.lib import get_conn, get_info_tab

conn = get_conn(resp_dir/'results.db')
info_tab = get_info_tab(conn)
trigger_time = info_tab['trigtimeMET'][0]
print(info_tab)
print(f"
Trigger time (MET seconds): {trigger_time:.3f}")


## 3) Quick preview of the event file
We only load a narrow time window around the trigger to keep memory and runtime low.

In [ ]:
from astropy.io import fits
import matplotlib.pyplot as plt
import numpy as np

PREVIEW_WINDOW_SEC = 30.0
PREVIEW_MIN_ENERGY_KEV = 14.0
PREVIEW_MAX_ENERGY_KEV = 150.0

ev_data = fits.open(resp_dir/'filter_evdata.fits')[1].data

# center 60 seconds around the trigger for a quick-look light curve
preview_mask = (ev_data['TIME'] > trigger_time - PREVIEW_WINDOW_SEC) & (ev_data['TIME'] < trigger_time + PREVIEW_WINDOW_SEC) & (ev_data['ENERGY'] > PREVIEW_MIN_ENERGY_KEV) & (ev_data['ENERGY'] < PREVIEW_MAX_ENERGY_KEV)
bin_edges = np.linspace(trigger_time - PREVIEW_WINDOW_SEC, trigger_time + PREVIEW_WINDOW_SEC, int(2 * PREVIEW_WINDOW_SEC) + 1)
counts, edges = np.histogram(ev_data['TIME'][preview_mask], bins=bin_edges)

plt.figure(figsize=(8,3))
plt.step(edges[:-1] - trigger_time, counts, where='post')
plt.xlabel('Time since trigger (s)')
plt.ylabel('Counts (15-150 keV)')
plt.title('Preview light curve around trigger')
plt.tight_layout()
plt.show()


The next notebook reuses the same `resp_dir` to run a full non-imaging likelihood fit and extract candidate properties.